# Semana 9 — Actividad de Sala: Prediccion de Demanda por Sucursal

**Curso:** 92-0030 Diagnostico y Predictibilidad
**Profesor:** Robin Sequeira
**Universidad:** ULACIT, I Cuatrimestre 2026

---

## Instrucciones para tu sala

**Este notebook ya esta resuelto. Ustedes NO tienen que programar nada.**

Lo unico que tienen que hacer es:

1. Buscar el numero de sala que les asignaron (del 1 al 10).
2. Escribir ese numero en la Celda 1, donde dice `NUMERO_DE_SALA`.
3. Correr todas las celdas de arriba hacia abajo (Run All, o una por una con Shift+Enter).
4. En cada celda donde aparezca el aviso **"INVESTIGUEN"**, deténganse: investiguen ese concepto (sílabo, apuntes, internet) y respondan la pregunta usando el resultado real que les dio a ustedes, no una definición genérica.
5. Al final, respondan las preguntas de cierre y armen su tríptico con todo lo que ya investigaron en el camino.

**Muy importante:** el código corre solo, pero los conceptos NO vienen explicados en el notebook. Cada resultado (la prueba ADF, los gráficos ACF y PACF, la tabla AIC, el modelo ARIMA, el pronóstico, el MAPE y el RMSE) trae al lado su propio aviso de investigación. Concepto y resultado van pegados, en el mismo lugar: no es "primero corro todo el notebook" y después, aparte, "investigo los conceptos". Es al revés: en cada parada, investigan el concepto y de inmediato lo aplican para interpretar lo que les salió a ustedes.

Cada sala tiene una sucursal distinta, con su propio comportamiento de ventas. Por eso los resultados de cada grupo van a ser diferentes, aunque el codigo sea el mismo.

**No hace falta subir ningun archivo a Databricks.** Los datos ya vienen incluidos en este notebook.


In [ ]:
# Celda 1: Aqui va el UNICO dato que ustedes cambian: el numero de su sala.
# No es programar, es solo escribir el numero que les toco (del 1 al 10).

NUMERO_DE_SALA = 1   # <-- CAMBIEN SOLO ESTE NUMERO POR EL DE SU SALA

# ── El resto de esta celda no se toca, solo se ejecuta ──
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import io
import warnings
warnings.filterwarnings("ignore")

from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error

COLOR_MORADO = "#3B1F5E"
COLOR_NARANJA = "#E8820C"
plt.rcParams["figure.figsize"] = (9, 5)

SUCURSAL_ASIGNADA = f"Sucursal_{NUMERO_DE_SALA}"
print(f"Sala configurada. Van a trabajar con: {SUCURSAL_ASIGNADA}")

## Celda 2 — Cargar los datos (ya vienen incluidos, no hay que subir nada)

In [ ]:
# Celda 2: El dataset completo de las 10 sucursales viene embebido aqui mismo.
# io.StringIO le dice a pandas "lee este texto como si fuera un archivo".

CSV_DATA = """Sucursal,Provincia,Fecha,Ventas
Sucursal_1,San Jose,2018-01-01,8663.0
Sucursal_1,San Jose,2018-02-01,8334.0
Sucursal_1,San Jose,2018-03-01,8468.0
Sucursal_1,San Jose,2018-04-01,8552.0
Sucursal_1,San Jose,2018-05-01,8322.0
Sucursal_1,San Jose,2018-06-01,8571.0
Sucursal_1,San Jose,2018-07-01,8420.0
Sucursal_1,San Jose,2018-08-01,7591.0
Sucursal_1,San Jose,2018-09-01,7855.0
Sucursal_1,San Jose,2018-10-01,7580.0
Sucursal_1,San Jose,2018-11-01,7504.0
Sucursal_1,San Jose,2018-12-01,8107.0
Sucursal_1,San Jose,2019-01-01,8726.0
Sucursal_1,San Jose,2019-02-01,8745.0
Sucursal_1,San Jose,2019-03-01,8759.0
Sucursal_1,San Jose,2019-04-01,8447.0
Sucursal_1,San Jose,2019-05-01,9018.0
Sucursal_1,San Jose,2019-06-01,8961.0
Sucursal_1,San Jose,2019-07-01,8849.0
Sucursal_1,San Jose,2019-08-01,8008.0
Sucursal_1,San Jose,2019-09-01,8373.0
Sucursal_1,San Jose,2019-10-01,7829.0
Sucursal_1,San Jose,2019-11-01,7924.0
Sucursal_1,San Jose,2019-12-01,9017.0
Sucursal_1,San Jose,2020-01-01,8949.0
Sucursal_1,San Jose,2020-02-01,8807.0
Sucursal_1,San Jose,2020-03-01,9078.0
Sucursal_1,San Jose,2020-04-01,8598.0
Sucursal_1,San Jose,2020-05-01,9502.0
Sucursal_1,San Jose,2020-06-01,9186.0
Sucursal_1,San Jose,2020-07-01,8954.0
Sucursal_1,San Jose,2020-08-01,9018.0
Sucursal_1,San Jose,2020-09-01,7908.0
Sucursal_1,San Jose,2020-10-01,8284.0
Sucursal_1,San Jose,2020-11-01,7864.0
Sucursal_1,San Jose,2020-12-01,8704.0
Sucursal_1,San Jose,2021-01-01,9019.0
Sucursal_1,San Jose,2021-02-01,9895.0
Sucursal_1,San Jose,2021-03-01,9981.0
Sucursal_1,San Jose,2021-04-01,9448.0
Sucursal_1,San Jose,2021-05-01,9810.0
Sucursal_1,San Jose,2021-06-01,9605.0
Sucursal_1,San Jose,2021-07-01,9642.0
Sucursal_1,San Jose,2021-08-01,8922.0
Sucursal_1,San Jose,2021-09-01,8253.0
Sucursal_1,San Jose,2021-10-01,8059.0
Sucursal_1,San Jose,2021-11-01,8836.0
Sucursal_1,San Jose,2021-12-01,9792.0
Sucursal_1,San Jose,2022-01-01,9747.0
Sucursal_1,San Jose,2022-02-01,9759.0
Sucursal_1,San Jose,2022-03-01,10378.0
Sucursal_1,San Jose,2022-04-01,9949.0
Sucursal_1,San Jose,2022-05-01,9985.0
Sucursal_1,San Jose,2022-06-01,10073.0
Sucursal_1,San Jose,2022-07-01,9827.0
Sucursal_1,San Jose,2022-08-01,9393.0
Sucursal_1,San Jose,2022-09-01,8682.0
Sucursal_1,San Jose,2022-10-01,8995.0
Sucursal_1,San Jose,2022-11-01,9077.0
Sucursal_1,San Jose,2022-12-01,9888.0
Sucursal_1,San Jose,2023-01-01,9948.0
Sucursal_1,San Jose,2023-02-01,9773.0
Sucursal_1,San Jose,2023-03-01,10235.0
Sucursal_1,San Jose,2023-04-01,10675.0
Sucursal_1,San Jose,2023-05-01,10224.0
Sucursal_1,San Jose,2023-06-01,10148.0
Sucursal_1,San Jose,2023-07-01,9922.0
Sucursal_1,San Jose,2023-08-01,9567.0
Sucursal_1,San Jose,2023-09-01,9325.0
Sucursal_1,San Jose,2023-10-01,8935.0
Sucursal_1,San Jose,2023-11-01,9835.0
Sucursal_1,San Jose,2023-12-01,9879.0
Sucursal_2,Alajuela,2018-01-01,9563.0
Sucursal_2,Alajuela,2018-02-01,10092.0
Sucursal_2,Alajuela,2018-03-01,9972.0
Sucursal_2,Alajuela,2018-04-01,9321.0
Sucursal_2,Alajuela,2018-05-01,9400.0
Sucursal_2,Alajuela,2018-06-01,9484.0
Sucursal_2,Alajuela,2018-07-01,9061.0
Sucursal_2,Alajuela,2018-08-01,8264.0
Sucursal_2,Alajuela,2018-09-01,8573.0
Sucursal_2,Alajuela,2018-10-01,8627.0
Sucursal_2,Alajuela,2018-11-01,8944.0
Sucursal_2,Alajuela,2018-12-01,9273.0
Sucursal_2,Alajuela,2019-01-01,10020.0
Sucursal_2,Alajuela,2019-02-01,10084.0
Sucursal_2,Alajuela,2019-03-01,10025.0
Sucursal_2,Alajuela,2019-04-01,10269.0
Sucursal_2,Alajuela,2019-05-01,10264.0
Sucursal_2,Alajuela,2019-06-01,10007.0
Sucursal_2,Alajuela,2019-07-01,9801.0
Sucursal_2,Alajuela,2019-08-01,9467.0
Sucursal_2,Alajuela,2019-09-01,8933.0
Sucursal_2,Alajuela,2019-10-01,8905.0
Sucursal_2,Alajuela,2019-11-01,9190.0
Sucursal_2,Alajuela,2019-12-01,10047.0
Sucursal_2,Alajuela,2020-01-01,11223.0
Sucursal_2,Alajuela,2020-02-01,11019.0
Sucursal_2,Alajuela,2020-03-01,10553.0
Sucursal_2,Alajuela,2020-04-01,10331.0
Sucursal_2,Alajuela,2020-05-01,10157.0
Sucursal_2,Alajuela,2020-06-01,10467.0
Sucursal_2,Alajuela,2020-07-01,10299.0
Sucursal_2,Alajuela,2020-08-01,9434.0
Sucursal_2,Alajuela,2020-09-01,9608.0
Sucursal_2,Alajuela,2020-10-01,9311.0
Sucursal_2,Alajuela,2020-11-01,10297.0
Sucursal_2,Alajuela,2020-12-01,10644.0
Sucursal_2,Alajuela,2021-01-01,11617.0
Sucursal_2,Alajuela,2021-02-01,11200.0
Sucursal_2,Alajuela,2021-03-01,11108.0
Sucursal_2,Alajuela,2021-04-01,11080.0
Sucursal_2,Alajuela,2021-05-01,11216.0
Sucursal_2,Alajuela,2021-06-01,11059.0
Sucursal_2,Alajuela,2021-07-01,10400.0
Sucursal_2,Alajuela,2021-08-01,10414.0
Sucursal_2,Alajuela,2021-09-01,10022.0
Sucursal_2,Alajuela,2021-10-01,9889.0
Sucursal_2,Alajuela,2021-11-01,10276.0
Sucursal_2,Alajuela,2021-12-01,11614.0
Sucursal_2,Alajuela,2022-01-01,11588.0
Sucursal_2,Alajuela,2022-02-01,11635.0
Sucursal_2,Alajuela,2022-03-01,11898.0
Sucursal_2,Alajuela,2022-04-01,11574.0
Sucursal_2,Alajuela,2022-05-01,11450.0
Sucursal_2,Alajuela,2022-06-01,11657.0
Sucursal_2,Alajuela,2022-07-01,11405.0
Sucursal_2,Alajuela,2022-08-01,11002.0
Sucursal_2,Alajuela,2022-09-01,10817.0
Sucursal_2,Alajuela,2022-10-01,10622.0
Sucursal_2,Alajuela,2022-11-01,10825.0
Sucursal_2,Alajuela,2022-12-01,11584.0
Sucursal_2,Alajuela,2023-01-01,12093.0
Sucursal_2,Alajuela,2023-02-01,12460.0
Sucursal_2,Alajuela,2023-03-01,11975.0
Sucursal_2,Alajuela,2023-04-01,11584.0
Sucursal_2,Alajuela,2023-05-01,12034.0
Sucursal_2,Alajuela,2023-06-01,11625.0
Sucursal_2,Alajuela,2023-07-01,11286.0
Sucursal_2,Alajuela,2023-08-01,11545.0
Sucursal_2,Alajuela,2023-09-01,10904.0
Sucursal_2,Alajuela,2023-10-01,11219.0
Sucursal_2,Alajuela,2023-11-01,11388.0
Sucursal_2,Alajuela,2023-12-01,11919.0
Sucursal_3,Cartago,2018-01-01,10482.0
Sucursal_3,Cartago,2018-02-01,10718.0
Sucursal_3,Cartago,2018-03-01,10419.0
Sucursal_3,Cartago,2018-04-01,9859.0
Sucursal_3,Cartago,2018-05-01,9830.0
Sucursal_3,Cartago,2018-06-01,9437.0
Sucursal_3,Cartago,2018-07-01,9793.0
Sucursal_3,Cartago,2018-08-01,9029.0
Sucursal_3,Cartago,2018-09-01,9309.0
Sucursal_3,Cartago,2018-10-01,9752.0
Sucursal_3,Cartago,2018-11-01,10509.0
Sucursal_3,Cartago,2018-12-01,11510.0
Sucursal_3,Cartago,2019-01-01,11112.0
Sucursal_3,Cartago,2019-02-01,11807.0
Sucursal_3,Cartago,2019-03-01,11568.0
Sucursal_3,Cartago,2019-04-01,11102.0
Sucursal_3,Cartago,2019-05-01,10362.0
Sucursal_3,Cartago,2019-06-01,10722.0
Sucursal_3,Cartago,2019-07-01,11049.0
Sucursal_3,Cartago,2019-08-01,9835.0
Sucursal_3,Cartago,2019-09-01,10224.0
Sucursal_3,Cartago,2019-10-01,10593.0
Sucursal_3,Cartago,2019-11-01,10710.0
Sucursal_3,Cartago,2019-12-01,11902.0
Sucursal_3,Cartago,2020-01-01,11972.0
Sucursal_3,Cartago,2020-02-01,12072.0
Sucursal_3,Cartago,2020-03-01,11702.0
Sucursal_3,Cartago,2020-04-01,11594.0
Sucursal_3,Cartago,2020-05-01,11325.0
Sucursal_3,Cartago,2020-06-01,11599.0
Sucursal_3,Cartago,2020-07-01,10956.0
Sucursal_3,Cartago,2020-08-01,10951.0
Sucursal_3,Cartago,2020-09-01,11084.0
Sucursal_3,Cartago,2020-10-01,11153.0
Sucursal_3,Cartago,2020-11-01,11248.0
Sucursal_3,Cartago,2020-12-01,12646.0
Sucursal_3,Cartago,2021-01-01,12864.0
Sucursal_3,Cartago,2021-02-01,13139.0
Sucursal_3,Cartago,2021-03-01,12746.0
Sucursal_3,Cartago,2021-04-01,12611.0
Sucursal_3,Cartago,2021-05-01,12105.0
Sucursal_3,Cartago,2021-06-01,11505.0
Sucursal_3,Cartago,2021-07-01,11552.0
Sucursal_3,Cartago,2021-08-01,11921.0
Sucursal_3,Cartago,2021-09-01,11451.0
Sucursal_3,Cartago,2021-10-01,12040.0
Sucursal_3,Cartago,2021-11-01,12432.0
Sucursal_3,Cartago,2021-12-01,12956.0
Sucursal_3,Cartago,2022-01-01,14037.0
Sucursal_3,Cartago,2022-02-01,13513.0
Sucursal_3,Cartago,2022-03-01,12783.0
Sucursal_3,Cartago,2022-04-01,13576.0
Sucursal_3,Cartago,2022-05-01,12238.0
Sucursal_3,Cartago,2022-06-01,12947.0
Sucursal_3,Cartago,2022-07-01,12941.0
Sucursal_3,Cartago,2022-08-01,12472.0
Sucursal_3,Cartago,2022-09-01,11757.0
Sucursal_3,Cartago,2022-10-01,12725.0
Sucursal_3,Cartago,2022-11-01,12654.0
Sucursal_3,Cartago,2022-12-01,13504.0
Sucursal_3,Cartago,2023-01-01,14338.0
Sucursal_3,Cartago,2023-02-01,14696.0
Sucursal_3,Cartago,2023-03-01,14185.0
Sucursal_3,Cartago,2023-04-01,13933.0
Sucursal_3,Cartago,2023-05-01,13383.0
Sucursal_3,Cartago,2023-06-01,13433.0
Sucursal_3,Cartago,2023-07-01,13059.0
Sucursal_3,Cartago,2023-08-01,13057.0
Sucursal_3,Cartago,2023-09-01,13138.0
Sucursal_3,Cartago,2023-10-01,13063.0
Sucursal_3,Cartago,2023-11-01,13930.0
Sucursal_3,Cartago,2023-12-01,14447.0
Sucursal_4,Heredia,2018-01-01,11858.0
Sucursal_4,Heredia,2018-02-01,11842.0
Sucursal_4,Heredia,2018-03-01,11135.0
Sucursal_4,Heredia,2018-04-01,10793.0
Sucursal_4,Heredia,2018-05-01,10443.0
Sucursal_4,Heredia,2018-06-01,11011.0
Sucursal_4,Heredia,2018-07-01,10508.0
Sucursal_4,Heredia,2018-08-01,10483.0
Sucursal_4,Heredia,2018-09-01,10248.0
Sucursal_4,Heredia,2018-10-01,10930.0
Sucursal_4,Heredia,2018-11-01,11890.0
Sucursal_4,Heredia,2018-12-01,12283.0
Sucursal_4,Heredia,2019-01-01,12926.0
Sucursal_4,Heredia,2019-02-01,12953.0
Sucursal_4,Heredia,2019-03-01,12056.0
Sucursal_4,Heredia,2019-04-01,11348.0
Sucursal_4,Heredia,2019-05-01,11590.0
Sucursal_4,Heredia,2019-06-01,11402.0
Sucursal_4,Heredia,2019-07-01,11599.0
Sucursal_4,Heredia,2019-08-01,11817.0
Sucursal_4,Heredia,2019-09-01,11825.0
Sucursal_4,Heredia,2019-10-01,11715.0
Sucursal_4,Heredia,2019-11-01,12415.0
Sucursal_4,Heredia,2019-12-01,13500.0
Sucursal_4,Heredia,2020-01-01,13800.0
Sucursal_4,Heredia,2020-02-01,13041.0
Sucursal_4,Heredia,2020-03-01,13391.0
Sucursal_4,Heredia,2020-04-01,12415.0
Sucursal_4,Heredia,2020-05-01,12416.0
Sucursal_4,Heredia,2020-06-01,11908.0
Sucursal_4,Heredia,2020-07-01,12776.0
Sucursal_4,Heredia,2020-08-01,12171.0
Sucursal_4,Heredia,2020-09-01,12337.0
Sucursal_4,Heredia,2020-10-01,12289.0
Sucursal_4,Heredia,2020-11-01,13426.0
Sucursal_4,Heredia,2020-12-01,14241.0
Sucursal_4,Heredia,2021-01-01,14684.0
Sucursal_4,Heredia,2021-02-01,14666.0
Sucursal_4,Heredia,2021-03-01,13914.0
Sucursal_4,Heredia,2021-04-01,13611.0
Sucursal_4,Heredia,2021-05-01,12943.0
Sucursal_4,Heredia,2021-06-01,12863.0
Sucursal_4,Heredia,2021-07-01,13270.0
Sucursal_4,Heredia,2021-08-01,13449.0
Sucursal_4,Heredia,2021-09-01,13656.0
Sucursal_4,Heredia,2021-10-01,13875.0
Sucursal_4,Heredia,2021-11-01,13831.0
Sucursal_4,Heredia,2021-12-01,15329.0
Sucursal_4,Heredia,2022-01-01,16094.0
Sucursal_4,Heredia,2022-02-01,15450.0
Sucursal_4,Heredia,2022-03-01,14636.0
Sucursal_4,Heredia,2022-04-01,14305.0
Sucursal_4,Heredia,2022-05-01,13832.0
Sucursal_4,Heredia,2022-06-01,14270.0
Sucursal_4,Heredia,2022-07-01,14172.0
Sucursal_4,Heredia,2022-08-01,14586.0
Sucursal_4,Heredia,2022-09-01,14325.0
Sucursal_4,Heredia,2022-10-01,14299.0
Sucursal_4,Heredia,2022-11-01,15465.0
Sucursal_4,Heredia,2022-12-01,16403.0
Sucursal_4,Heredia,2023-01-01,16365.0
Sucursal_4,Heredia,2023-02-01,16198.0
Sucursal_4,Heredia,2023-03-01,14981.0
Sucursal_4,Heredia,2023-04-01,15389.0
Sucursal_4,Heredia,2023-05-01,14706.0
Sucursal_4,Heredia,2023-06-01,14314.0
Sucursal_4,Heredia,2023-07-01,14528.0
Sucursal_4,Heredia,2023-08-01,14985.0
Sucursal_4,Heredia,2023-09-01,15306.0
Sucursal_4,Heredia,2023-10-01,15395.0
Sucursal_4,Heredia,2023-11-01,16324.0
Sucursal_4,Heredia,2023-12-01,16671.0
Sucursal_5,Guanacaste,2018-01-01,12571.0
Sucursal_5,Guanacaste,2018-02-01,12824.0
Sucursal_5,Guanacaste,2018-03-01,11828.0
Sucursal_5,Guanacaste,2018-04-01,10978.0
Sucursal_5,Guanacaste,2018-05-01,11287.0
Sucursal_5,Guanacaste,2018-06-01,10877.0
Sucursal_5,Guanacaste,2018-07-01,11451.0
Sucursal_5,Guanacaste,2018-08-01,12159.0
Sucursal_5,Guanacaste,2018-09-01,12609.0
Sucursal_5,Guanacaste,2018-10-01,12430.0
Sucursal_5,Guanacaste,2018-11-01,13718.0
Sucursal_5,Guanacaste,2018-12-01,13682.0
Sucursal_5,Guanacaste,2019-01-01,14203.0
Sucursal_5,Guanacaste,2019-02-01,13554.0
Sucursal_5,Guanacaste,2019-03-01,12222.0
Sucursal_5,Guanacaste,2019-04-01,12154.0
Sucursal_5,Guanacaste,2019-05-01,12252.0
Sucursal_5,Guanacaste,2019-06-01,12222.0
Sucursal_5,Guanacaste,2019-07-01,12647.0
Sucursal_5,Guanacaste,2019-08-01,13360.0
Sucursal_5,Guanacaste,2019-09-01,12670.0
Sucursal_5,Guanacaste,2019-10-01,13013.0
Sucursal_5,Guanacaste,2019-11-01,14028.0
Sucursal_5,Guanacaste,2019-12-01,14566.0
Sucursal_5,Guanacaste,2020-01-01,15618.0
Sucursal_5,Guanacaste,2020-02-01,14794.0
Sucursal_5,Guanacaste,2020-03-01,13541.0
Sucursal_5,Guanacaste,2020-04-01,12946.0
Sucursal_5,Guanacaste,2020-05-01,13968.0
Sucursal_5,Guanacaste,2020-06-01,13149.0
Sucursal_5,Guanacaste,2020-07-01,13498.0
Sucursal_5,Guanacaste,2020-08-01,14081.0
Sucursal_5,Guanacaste,2020-09-01,14373.0
Sucursal_5,Guanacaste,2020-10-01,14661.0
Sucursal_5,Guanacaste,2020-11-01,15024.0
Sucursal_5,Guanacaste,2020-12-01,15659.0
Sucursal_5,Guanacaste,2021-01-01,16451.0
Sucursal_5,Guanacaste,2021-02-01,15754.0
Sucursal_5,Guanacaste,2021-03-01,14364.0
Sucursal_5,Guanacaste,2021-04-01,14615.0
Sucursal_5,Guanacaste,2021-05-01,14360.0
Sucursal_5,Guanacaste,2021-06-01,14351.0
Sucursal_5,Guanacaste,2021-07-01,14744.0
Sucursal_5,Guanacaste,2021-08-01,15137.0
Sucursal_5,Guanacaste,2021-09-01,15015.0
Sucursal_5,Guanacaste,2021-10-01,15222.0
Sucursal_5,Guanacaste,2021-11-01,16511.0
Sucursal_5,Guanacaste,2021-12-01,16646.0
Sucursal_5,Guanacaste,2022-01-01,17463.0
Sucursal_5,Guanacaste,2022-02-01,16376.0
Sucursal_5,Guanacaste,2022-03-01,15953.0
Sucursal_5,Guanacaste,2022-04-01,15264.0
Sucursal_5,Guanacaste,2022-05-01,15772.0
Sucursal_5,Guanacaste,2022-06-01,15160.0
Sucursal_5,Guanacaste,2022-07-01,15443.0
Sucursal_5,Guanacaste,2022-08-01,15927.0
Sucursal_5,Guanacaste,2022-09-01,15951.0
Sucursal_5,Guanacaste,2022-10-01,16553.0
Sucursal_5,Guanacaste,2022-11-01,17623.0
Sucursal_5,Guanacaste,2022-12-01,18456.0
Sucursal_5,Guanacaste,2023-01-01,18962.0
Sucursal_5,Guanacaste,2023-02-01,17689.0
Sucursal_5,Guanacaste,2023-03-01,17937.0
Sucursal_5,Guanacaste,2023-04-01,17043.0
Sucursal_5,Guanacaste,2023-05-01,16197.0
Sucursal_5,Guanacaste,2023-06-01,15835.0
Sucursal_5,Guanacaste,2023-07-01,16963.0
Sucursal_5,Guanacaste,2023-08-01,17244.0
Sucursal_5,Guanacaste,2023-09-01,16643.0
Sucursal_5,Guanacaste,2023-10-01,17852.0
Sucursal_5,Guanacaste,2023-11-01,18498.0
Sucursal_5,Guanacaste,2023-12-01,19022.0
Sucursal_6,Puntarenas,2018-01-01,13917.0
Sucursal_6,Puntarenas,2018-02-01,13338.0
Sucursal_6,Puntarenas,2018-03-01,11826.0
Sucursal_6,Puntarenas,2018-04-01,11454.0
Sucursal_6,Puntarenas,2018-05-01,11259.0
Sucursal_6,Puntarenas,2018-06-01,11887.0
Sucursal_6,Puntarenas,2018-07-01,12920.0
Sucursal_6,Puntarenas,2018-08-01,13517.0
Sucursal_6,Puntarenas,2018-09-01,13376.0
Sucursal_6,Puntarenas,2018-10-01,13152.0
Sucursal_6,Puntarenas,2018-11-01,14707.0
Sucursal_6,Puntarenas,2018-12-01,14439.0
Sucursal_6,Puntarenas,2019-01-01,15001.0
Sucursal_6,Puntarenas,2019-02-01,14869.0
Sucursal_6,Puntarenas,2019-03-01,13251.0
Sucursal_6,Puntarenas,2019-04-01,12217.0
Sucursal_6,Puntarenas,2019-05-01,12237.0
Sucursal_6,Puntarenas,2019-06-01,13832.0
Sucursal_6,Puntarenas,2019-07-01,14146.0
Sucursal_6,Puntarenas,2019-08-01,14783.0
Sucursal_6,Puntarenas,2019-09-01,15158.0
Sucursal_6,Puntarenas,2019-10-01,15169.0
Sucursal_6,Puntarenas,2019-11-01,15654.0
Sucursal_6,Puntarenas,2019-12-01,15926.0
Sucursal_6,Puntarenas,2020-01-01,15139.0
Sucursal_6,Puntarenas,2020-02-01,14975.0
Sucursal_6,Puntarenas,2020-03-01,14492.0
Sucursal_6,Puntarenas,2020-04-01,13832.0
Sucursal_6,Puntarenas,2020-05-01,14150.0
Sucursal_6,Puntarenas,2020-06-01,14638.0
Sucursal_6,Puntarenas,2020-07-01,15468.0
Sucursal_6,Puntarenas,2020-08-01,16129.0
Sucursal_6,Puntarenas,2020-09-01,15784.0
Sucursal_6,Puntarenas,2020-10-01,16346.0
Sucursal_6,Puntarenas,2020-11-01,16459.0
Sucursal_6,Puntarenas,2020-12-01,17858.0
Sucursal_6,Puntarenas,2021-01-01,17492.0
Sucursal_6,Puntarenas,2021-02-01,16938.0
Sucursal_6,Puntarenas,2021-03-01,16316.0
Sucursal_6,Puntarenas,2021-04-01,15531.0
Sucursal_6,Puntarenas,2021-05-01,15247.0
Sucursal_6,Puntarenas,2021-06-01,16218.0
Sucursal_6,Puntarenas,2021-07-01,16171.0
Sucursal_6,Puntarenas,2021-08-01,17266.0
Sucursal_6,Puntarenas,2021-09-01,17402.0
Sucursal_6,Puntarenas,2021-10-01,17140.0
Sucursal_6,Puntarenas,2021-11-01,18750.0
Sucursal_6,Puntarenas,2021-12-01,18531.0
Sucursal_6,Puntarenas,2022-01-01,18977.0
Sucursal_6,Puntarenas,2022-02-01,17879.0
Sucursal_6,Puntarenas,2022-03-01,17384.0
Sucursal_6,Puntarenas,2022-04-01,16873.0
Sucursal_6,Puntarenas,2022-05-01,16238.0
Sucursal_6,Puntarenas,2022-06-01,16667.0
Sucursal_6,Puntarenas,2022-07-01,17886.0
Sucursal_6,Puntarenas,2022-08-01,18671.0
Sucursal_6,Puntarenas,2022-09-01,18484.0
Sucursal_6,Puntarenas,2022-10-01,18752.0
Sucursal_6,Puntarenas,2022-11-01,19713.0
Sucursal_6,Puntarenas,2022-12-01,19071.0
Sucursal_6,Puntarenas,2023-01-01,19766.0
Sucursal_6,Puntarenas,2023-02-01,18802.0
Sucursal_6,Puntarenas,2023-03-01,18400.0
Sucursal_6,Puntarenas,2023-04-01,17416.0
Sucursal_6,Puntarenas,2023-05-01,17398.0
Sucursal_6,Puntarenas,2023-06-01,18339.0
Sucursal_6,Puntarenas,2023-07-01,18678.0
Sucursal_6,Puntarenas,2023-08-01,19629.0
Sucursal_6,Puntarenas,2023-09-01,19635.0
Sucursal_6,Puntarenas,2023-10-01,20094.0
Sucursal_6,Puntarenas,2023-11-01,20551.0
Sucursal_6,Puntarenas,2023-12-01,21617.0
Sucursal_7,Limon,2018-01-01,14074.0
Sucursal_7,Limon,2018-02-01,13486.0
Sucursal_7,Limon,2018-03-01,12515.0
Sucursal_7,Limon,2018-04-01,11627.0
Sucursal_7,Limon,2018-05-01,12602.0
Sucursal_7,Limon,2018-06-01,13578.0
Sucursal_7,Limon,2018-07-01,14578.0
Sucursal_7,Limon,2018-08-01,14849.0
Sucursal_7,Limon,2018-09-01,14977.0
Sucursal_7,Limon,2018-10-01,15549.0
Sucursal_7,Limon,2018-11-01,16368.0
Sucursal_7,Limon,2018-12-01,15492.0
Sucursal_7,Limon,2019-01-01,15752.0
Sucursal_7,Limon,2019-02-01,14596.0
Sucursal_7,Limon,2019-03-01,14068.0
Sucursal_7,Limon,2019-04-01,13527.0
Sucursal_7,Limon,2019-05-01,14223.0
Sucursal_7,Limon,2019-06-01,13855.0
Sucursal_7,Limon,2019-07-01,15679.0
Sucursal_7,Limon,2019-08-01,16070.0
Sucursal_7,Limon,2019-09-01,16535.0
Sucursal_7,Limon,2019-10-01,16431.0
Sucursal_7,Limon,2019-11-01,16764.0
Sucursal_7,Limon,2019-12-01,17491.0
Sucursal_7,Limon,2020-01-01,17698.0
Sucursal_7,Limon,2020-02-01,15127.0
Sucursal_7,Limon,2020-03-01,15103.0
Sucursal_7,Limon,2020-04-01,14921.0
Sucursal_7,Limon,2020-05-01,15273.0
Sucursal_7,Limon,2020-06-01,16202.0
Sucursal_7,Limon,2020-07-01,16841.0
Sucursal_7,Limon,2020-08-01,17527.0
Sucursal_7,Limon,2020-09-01,18283.0
Sucursal_7,Limon,2020-10-01,18537.0
Sucursal_7,Limon,2020-11-01,18477.0
Sucursal_7,Limon,2020-12-01,18369.0
Sucursal_7,Limon,2021-01-01,18470.0
Sucursal_7,Limon,2021-02-01,17523.0
Sucursal_7,Limon,2021-03-01,16859.0
Sucursal_7,Limon,2021-04-01,16289.0
Sucursal_7,Limon,2021-05-01,16481.0
Sucursal_7,Limon,2021-06-01,17840.0
Sucursal_7,Limon,2021-07-01,18345.0
Sucursal_7,Limon,2021-08-01,19619.0
Sucursal_7,Limon,2021-09-01,19892.0
Sucursal_7,Limon,2021-10-01,19143.0
Sucursal_7,Limon,2021-11-01,20037.0
Sucursal_7,Limon,2021-12-01,19752.0
Sucursal_7,Limon,2022-01-01,20271.0
Sucursal_7,Limon,2022-02-01,19386.0
Sucursal_7,Limon,2022-03-01,18012.0
Sucursal_7,Limon,2022-04-01,18053.0
Sucursal_7,Limon,2022-05-01,17992.0
Sucursal_7,Limon,2022-06-01,18971.0
Sucursal_7,Limon,2022-07-01,20011.0
Sucursal_7,Limon,2022-08-01,20704.0
Sucursal_7,Limon,2022-09-01,20775.0
Sucursal_7,Limon,2022-10-01,21076.0
Sucursal_7,Limon,2022-11-01,21093.0
Sucursal_7,Limon,2022-12-01,21842.0
Sucursal_7,Limon,2023-01-01,21692.0
Sucursal_7,Limon,2023-02-01,20308.0
Sucursal_7,Limon,2023-03-01,19009.0
Sucursal_7,Limon,2023-04-01,19393.0
Sucursal_7,Limon,2023-05-01,20012.0
Sucursal_7,Limon,2023-06-01,20589.0
Sucursal_7,Limon,2023-07-01,21186.0
Sucursal_7,Limon,2023-08-01,22295.0
Sucursal_7,Limon,2023-09-01,22899.0
Sucursal_7,Limon,2023-10-01,21765.0
Sucursal_7,Limon,2023-11-01,22938.0
Sucursal_7,Limon,2023-12-01,22883.0
Sucursal_8,Perez Zeledon,2018-01-01,14037.0
Sucursal_8,Perez Zeledon,2018-02-01,13806.0
Sucursal_8,Perez Zeledon,2018-03-01,12355.0
Sucursal_8,Perez Zeledon,2018-04-01,12896.0
Sucursal_8,Perez Zeledon,2018-05-01,14149.0
Sucursal_8,Perez Zeledon,2018-06-01,15095.0
Sucursal_8,Perez Zeledon,2018-07-01,15849.0
Sucursal_8,Perez Zeledon,2018-08-01,15951.0
Sucursal_8,Perez Zeledon,2018-09-01,16012.0
Sucursal_8,Perez Zeledon,2018-10-01,15569.0
Sucursal_8,Perez Zeledon,2018-11-01,16901.0
Sucursal_8,Perez Zeledon,2018-12-01,16212.0
Sucursal_8,Perez Zeledon,2019-01-01,15583.0
Sucursal_8,Perez Zeledon,2019-02-01,14725.0
Sucursal_8,Perez Zeledon,2019-03-01,14065.0
Sucursal_8,Perez Zeledon,2019-04-01,14270.0
Sucursal_8,Perez Zeledon,2019-05-01,14757.0
Sucursal_8,Perez Zeledon,2019-06-01,17300.0
Sucursal_8,Perez Zeledon,2019-07-01,17520.0
Sucursal_8,Perez Zeledon,2019-08-01,17959.0
Sucursal_8,Perez Zeledon,2019-09-01,18252.0
Sucursal_8,Perez Zeledon,2019-10-01,18473.0
Sucursal_8,Perez Zeledon,2019-11-01,18414.0
Sucursal_8,Perez Zeledon,2019-12-01,17740.0
Sucursal_8,Perez Zeledon,2020-01-01,17710.0
Sucursal_8,Perez Zeledon,2020-02-01,16910.0
Sucursal_8,Perez Zeledon,2020-03-01,16308.0
Sucursal_8,Perez Zeledon,2020-04-01,16120.0
Sucursal_8,Perez Zeledon,2020-05-01,16509.0
Sucursal_8,Perez Zeledon,2020-06-01,17775.0
Sucursal_8,Perez Zeledon,2020-07-01,19500.0
Sucursal_8,Perez Zeledon,2020-08-01,19423.0
Sucursal_8,Perez Zeledon,2020-09-01,20284.0
Sucursal_8,Perez Zeledon,2020-10-01,19543.0
Sucursal_8,Perez Zeledon,2020-11-01,19117.0
Sucursal_8,Perez Zeledon,2020-12-01,19424.0
Sucursal_8,Perez Zeledon,2021-01-01,19033.0
Sucursal_8,Perez Zeledon,2021-02-01,18420.0
Sucursal_8,Perez Zeledon,2021-03-01,18090.0
Sucursal_8,Perez Zeledon,2021-04-01,17077.0
Sucursal_8,Perez Zeledon,2021-05-01,17768.0
Sucursal_8,Perez Zeledon,2021-06-01,20018.0
Sucursal_8,Perez Zeledon,2021-07-01,20982.0
Sucursal_8,Perez Zeledon,2021-08-01,21568.0
Sucursal_8,Perez Zeledon,2021-09-01,21546.0
Sucursal_8,Perez Zeledon,2021-10-01,21077.0
Sucursal_8,Perez Zeledon,2021-11-01,21449.0
Sucursal_8,Perez Zeledon,2021-12-01,20822.0
Sucursal_8,Perez Zeledon,2022-01-01,20876.0
Sucursal_8,Perez Zeledon,2022-02-01,19767.0
Sucursal_8,Perez Zeledon,2022-03-01,18901.0
Sucursal_8,Perez Zeledon,2022-04-01,19501.0
Sucursal_8,Perez Zeledon,2022-05-01,20339.0
Sucursal_8,Perez Zeledon,2022-06-01,20749.0
Sucursal_8,Perez Zeledon,2022-07-01,22234.0
Sucursal_8,Perez Zeledon,2022-08-01,23239.0
Sucursal_8,Perez Zeledon,2022-09-01,23410.0
Sucursal_8,Perez Zeledon,2022-10-01,22508.0
Sucursal_8,Perez Zeledon,2022-11-01,22686.0
Sucursal_8,Perez Zeledon,2022-12-01,22239.0
Sucursal_8,Perez Zeledon,2023-01-01,23450.0
Sucursal_8,Perez Zeledon,2023-02-01,21897.0
Sucursal_8,Perez Zeledon,2023-03-01,21311.0
Sucursal_8,Perez Zeledon,2023-04-01,21137.0
Sucursal_8,Perez Zeledon,2023-05-01,22101.0
Sucursal_8,Perez Zeledon,2023-06-01,23363.0
Sucursal_8,Perez Zeledon,2023-07-01,24160.0
Sucursal_8,Perez Zeledon,2023-08-01,25035.0
Sucursal_8,Perez Zeledon,2023-09-01,24643.0
Sucursal_8,Perez Zeledon,2023-10-01,24893.0
Sucursal_8,Perez Zeledon,2023-11-01,24090.0
Sucursal_8,Perez Zeledon,2023-12-01,24199.0
Sucursal_9,Turrialba,2018-01-01,14914.0
Sucursal_9,Turrialba,2018-02-01,14654.0
Sucursal_9,Turrialba,2018-03-01,13774.0
Sucursal_9,Turrialba,2018-04-01,13714.0
Sucursal_9,Turrialba,2018-05-01,15146.0
Sucursal_9,Turrialba,2018-06-01,16570.0
Sucursal_9,Turrialba,2018-07-01,18682.0
Sucursal_9,Turrialba,2018-08-01,18463.0
Sucursal_9,Turrialba,2018-09-01,18080.0
Sucursal_9,Turrialba,2018-10-01,17069.0
Sucursal_9,Turrialba,2018-11-01,16453.0
Sucursal_9,Turrialba,2018-12-01,17130.0
Sucursal_9,Turrialba,2019-01-01,17122.0
Sucursal_9,Turrialba,2019-02-01,16168.0
Sucursal_9,Turrialba,2019-03-01,15505.0
Sucursal_9,Turrialba,2019-04-01,16250.0
Sucursal_9,Turrialba,2019-05-01,16843.0
Sucursal_9,Turrialba,2019-06-01,18888.0
Sucursal_9,Turrialba,2019-07-01,19413.0
Sucursal_9,Turrialba,2019-08-01,19496.0
Sucursal_9,Turrialba,2019-09-01,19594.0
Sucursal_9,Turrialba,2019-10-01,18884.0
Sucursal_9,Turrialba,2019-11-01,19262.0
Sucursal_9,Turrialba,2019-12-01,18335.0
Sucursal_9,Turrialba,2020-01-01,18253.0
Sucursal_9,Turrialba,2020-02-01,17262.0
Sucursal_9,Turrialba,2020-03-01,16548.0
Sucursal_9,Turrialba,2020-04-01,16876.0
Sucursal_9,Turrialba,2020-05-01,18985.0
Sucursal_9,Turrialba,2020-06-01,19892.0
Sucursal_9,Turrialba,2020-07-01,21967.0
Sucursal_9,Turrialba,2020-08-01,22277.0
Sucursal_9,Turrialba,2020-09-01,21044.0
Sucursal_9,Turrialba,2020-10-01,21226.0
Sucursal_9,Turrialba,2020-11-01,20374.0
Sucursal_9,Turrialba,2020-12-01,20711.0
Sucursal_9,Turrialba,2021-01-01,19951.0
Sucursal_9,Turrialba,2021-02-01,19923.0
Sucursal_9,Turrialba,2021-03-01,18728.0
Sucursal_9,Turrialba,2021-04-01,19662.0
Sucursal_9,Turrialba,2021-05-01,20407.0
Sucursal_9,Turrialba,2021-06-01,22121.0
Sucursal_9,Turrialba,2021-07-01,23564.0
Sucursal_9,Turrialba,2021-08-01,23565.0
Sucursal_9,Turrialba,2021-09-01,22931.0
Sucursal_9,Turrialba,2021-10-01,22644.0
Sucursal_9,Turrialba,2021-11-01,22410.0
Sucursal_9,Turrialba,2021-12-01,21482.0
Sucursal_9,Turrialba,2022-01-01,22073.0
Sucursal_9,Turrialba,2022-02-01,21605.0
Sucursal_9,Turrialba,2022-03-01,20606.0
Sucursal_9,Turrialba,2022-04-01,21364.0
Sucursal_9,Turrialba,2022-05-01,21852.0
Sucursal_9,Turrialba,2022-06-01,24565.0
Sucursal_9,Turrialba,2022-07-01,25007.0
Sucursal_9,Turrialba,2022-08-01,25262.0
Sucursal_9,Turrialba,2022-09-01,25252.0
Sucursal_9,Turrialba,2022-10-01,24087.0
Sucursal_9,Turrialba,2022-11-01,24126.0
Sucursal_9,Turrialba,2022-12-01,23420.0
Sucursal_9,Turrialba,2023-01-01,23410.0
Sucursal_9,Turrialba,2023-02-01,22559.0
Sucursal_9,Turrialba,2023-03-01,22629.0
Sucursal_9,Turrialba,2023-04-01,23429.0
Sucursal_9,Turrialba,2023-05-01,24287.0
Sucursal_9,Turrialba,2023-06-01,26137.0
Sucursal_9,Turrialba,2023-07-01,26484.0
Sucursal_9,Turrialba,2023-08-01,26732.0
Sucursal_9,Turrialba,2023-09-01,26371.0
Sucursal_9,Turrialba,2023-10-01,26180.0
Sucursal_9,Turrialba,2023-11-01,25827.0
Sucursal_9,Turrialba,2023-12-01,25643.0
Sucursal_10,Liberia,2018-01-01,15397.0
Sucursal_10,Liberia,2018-02-01,14937.0
Sucursal_10,Liberia,2018-03-01,14957.0
Sucursal_10,Liberia,2018-04-01,16130.0
Sucursal_10,Liberia,2018-05-01,16686.0
Sucursal_10,Liberia,2018-06-01,18585.0
Sucursal_10,Liberia,2018-07-01,20097.0
Sucursal_10,Liberia,2018-08-01,19234.0
Sucursal_10,Liberia,2018-09-01,18516.0
Sucursal_10,Liberia,2018-10-01,17006.0
Sucursal_10,Liberia,2018-11-01,17741.0
Sucursal_10,Liberia,2018-12-01,16038.0
Sucursal_10,Liberia,2019-01-01,16622.0
Sucursal_10,Liberia,2019-02-01,16529.0
Sucursal_10,Liberia,2019-03-01,17643.0
Sucursal_10,Liberia,2019-04-01,17700.0
Sucursal_10,Liberia,2019-05-01,18806.0
Sucursal_10,Liberia,2019-06-01,19935.0
Sucursal_10,Liberia,2019-07-01,21541.0
Sucursal_10,Liberia,2019-08-01,21219.0
Sucursal_10,Liberia,2019-09-01,19894.0
Sucursal_10,Liberia,2019-10-01,18606.0
Sucursal_10,Liberia,2019-11-01,18405.0
Sucursal_10,Liberia,2019-12-01,19078.0
Sucursal_10,Liberia,2020-01-01,19223.0
Sucursal_10,Liberia,2020-02-01,18993.0
Sucursal_10,Liberia,2020-03-01,18861.0
Sucursal_10,Liberia,2020-04-01,18768.0
Sucursal_10,Liberia,2020-05-01,20919.0
Sucursal_10,Liberia,2020-06-01,22022.0
Sucursal_10,Liberia,2020-07-01,23382.0
Sucursal_10,Liberia,2020-08-01,23350.0
Sucursal_10,Liberia,2020-09-01,21881.0
Sucursal_10,Liberia,2020-10-01,20787.0
Sucursal_10,Liberia,2020-11-01,21287.0
Sucursal_10,Liberia,2020-12-01,21117.0
Sucursal_10,Liberia,2021-01-01,20796.0
Sucursal_10,Liberia,2021-02-01,19652.0
Sucursal_10,Liberia,2021-03-01,20885.0
Sucursal_10,Liberia,2021-04-01,21779.0
Sucursal_10,Liberia,2021-05-01,23361.0
Sucursal_10,Liberia,2021-06-01,24155.0
Sucursal_10,Liberia,2021-07-01,25068.0
Sucursal_10,Liberia,2021-08-01,24940.0
Sucursal_10,Liberia,2021-09-01,24512.0
Sucursal_10,Liberia,2021-10-01,22702.0
Sucursal_10,Liberia,2021-11-01,22791.0
Sucursal_10,Liberia,2021-12-01,22548.0
Sucursal_10,Liberia,2022-01-01,23165.0
Sucursal_10,Liberia,2022-02-01,22253.0
Sucursal_10,Liberia,2022-03-01,23048.0
Sucursal_10,Liberia,2022-04-01,23275.0
Sucursal_10,Liberia,2022-05-01,24762.0
Sucursal_10,Liberia,2022-06-01,26652.0
Sucursal_10,Liberia,2022-07-01,27816.0
Sucursal_10,Liberia,2022-08-01,26381.0
Sucursal_10,Liberia,2022-09-01,26563.0
Sucursal_10,Liberia,2022-10-01,25427.0
Sucursal_10,Liberia,2022-11-01,24693.0
Sucursal_10,Liberia,2022-12-01,25530.0
Sucursal_10,Liberia,2023-01-01,25111.0
Sucursal_10,Liberia,2023-02-01,25458.0
Sucursal_10,Liberia,2023-03-01,24611.0
Sucursal_10,Liberia,2023-04-01,25096.0
Sucursal_10,Liberia,2023-05-01,27365.0
Sucursal_10,Liberia,2023-06-01,29337.0
Sucursal_10,Liberia,2023-07-01,29577.0
Sucursal_10,Liberia,2023-08-01,29474.0
Sucursal_10,Liberia,2023-09-01,27908.0
Sucursal_10,Liberia,2023-10-01,26597.0
Sucursal_10,Liberia,2023-11-01,26702.0
Sucursal_10,Liberia,2023-12-01,27551.0
"""

df_todas = pd.read_csv(io.StringIO(CSV_DATA))
df_todas["Fecha"] = pd.to_datetime(df_todas["Fecha"])

print("Dataset completo cargado correctamente.")
print(f"Total de filas: {df_todas.shape[0]}  (10 sucursales x 72 meses cada una)")
print()
print("Sucursales disponibles:")
print(df_todas["Sucursal"].unique().tolist())
print()

# Filtramos SOLO la sucursal de esta sala
df = df_todas[df_todas["Sucursal"] == SUCURSAL_ASIGNADA].copy()
df = df.sort_values("Fecha").set_index("Fecha")
provincia = df["Provincia"].iloc[0]

print(f"Trabajando con: {SUCURSAL_ASIGNADA} (provincia: {provincia})")
print(f"Meses de historia disponibles: {len(df)}")
print()
print("Interpretacion: cada sala tiene una sucursal distinta.")
print("Los numeros que les van a salir de aqui en adelante son UNICOS de su caso,")
print("no van a ser iguales a los de otra sala. Esa es la idea de la actividad.")

## Celda 3 — Conocer la serie: primer vistazo

In [ ]:
# Celda 3: Miremos como se ven los datos antes de hacer cualquier analisis.

print("=" * 55)
print(f"PRIMERAS FILAS - {SUCURSAL_ASIGNADA}")
print("=" * 55)
display(df.head())

print()
print(f"Venta promedio mensual: {df['Ventas'].mean():,.0f}")
print(f"Venta minima registrada: {df['Ventas'].min():,.0f}")
print(f"Venta maxima registrada: {df['Ventas'].max():,.0f}")

nulos = df["Ventas"].isnull().sum()
print(f"\nValores nulos: {nulos}")
if nulos == 0:
    print("La serie esta completa, sin huecos. Podemos seguir.")

## Celda 4 — Graficar la serie completa

In [ ]:
# Celda 4: Un grafico dice mas que mil numeros. Veamos la evolucion de las ventas.

plt.figure(figsize=(11, 5))
plt.plot(df.index, df["Ventas"], color=COLOR_MORADO, linewidth=2)
plt.title(f"Ventas mensuales - {SUCURSAL_ASIGNADA} ({provincia})", fontsize=14)
plt.xlabel("Fecha")
plt.ylabel("Ventas")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("PREGUNTA PARA EL GRUPO (anotenla, la van a usar en el triptico):")
print("Mirando el grafico, sin calcular nada todavia: ¿la serie parece tener una")
print("tendencia clara (sube o baja con el tiempo)? ¿Ven algun patron que se repita")
print("cada cierta cantidad de meses?")

## Celda 5 — ¿Es estacionaria la serie? La prueba ADF

In [ ]:
# Celda 5: Antes de modelar, necesitamos saber si la serie es estacionaria.
# Usamos la prueba ADF (Augmented Dickey-Fuller). Ya viene lista, solo se ejecuta.

resultado_adf = adfuller(df["Ventas"].dropna())
estadistico = resultado_adf[0]
p_value = resultado_adf[1]

print(f"Sucursal analizada: {SUCURSAL_ASIGNADA}")
print(f"Estadistico ADF: {estadistico:.4f}")
print(f"p-value: {p_value:.4f}")
print()

if p_value < 0.05:
    es_estacionaria = True
    print("Resultado: la serie SI es estacionaria (p-value < 0.05).")
    print("Interpretacion: no hace falta diferenciar la serie. Usamos d = 0.")
else:
    es_estacionaria = False
    print("Resultado: la serie NO es estacionaria (p-value >= 0.05).")
    print("Interpretacion: hay que diferenciar la serie una vez. Usamos d = 1.")

d_sugerido = 0 if es_estacionaria else 1
print()
print(f"Parametro d sugerido para el modelo ARIMA de esta sucursal: d = {d_sugerido}")
print()
print("PREGUNTA PARA EL GRUPO:")
print("¿Por que creen que su sucursal dio ese resultado? Piensen en si su grafico")
print("de la Celda 4 mostraba una tendencia fuerte o no.")

### 🔎 INVESTIGUEN: ¿qué es la prueba ADF?

Antes de seguir, investiguen (sílabo, apuntes, internet): **¿qué es la prueba ADF (Augmented Dickey-Fuller) y para qué sirve?**

Ahora respondan usando SU propio resultado, el que salió arriba para su sucursal:

- Con sus propias palabras, ¿qué significa el p-value que les dio a ustedes?
- Según ese p-value, ¿su serie es estacionaria o no? ¿Qué relación tiene esto con el parámetro d que se usó después?

No copien la definición de internet tal cual: expliquen como si se lo contaran a un compañero que faltó a clase, usando el número real que obtuvieron.


## Celda 6 — ACF y PACF: la pista de p y q

In [ ]:
# Celda 6: Estos graficos nos ayudan a elegir p y q para el modelo ARIMA.
# Si la serie no era estacionaria, trabajamos sobre la serie diferenciada.

serie_para_graficos = df["Ventas"].diff().dropna() if not es_estacionaria else df["Ventas"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.5))
plot_acf(serie_para_graficos, ax=ax1, lags=24, color=COLOR_MORADO)
ax1.set_title("ACF (ayuda a elegir q)")
plot_pacf(serie_para_graficos, ax=ax2, lags=24, color=COLOR_NARANJA)
ax2.set_title("PACF (ayuda a elegir p)")
plt.tight_layout()
plt.show()

print("Como leerlo: busquen en que punto (lag) las barras dejan de salirse")
print("de la zona sombreada de forma clara. Ese punto es una pista del valor")
print("de p (mirando el PACF) y de q (mirando el ACF).")
print()
print("PREGUNTA PARA EL GRUPO:")
print("¿En que lag aproximado dejan de verse barras importantes en cada grafico?")
print("No tiene que ser exacto, es para que practiquen la lectura visual.")

### 🔎 INVESTIGUEN: ¿qué son ACF y PACF?

Investiguen: **¿qué es la autocorrelación (ACF) y qué es la autocorrelación parcial (PACF)? ¿En qué se diferencian?**

Ahora conecten esto con SU gráfico de arriba:

- ¿En qué lag su ACF deja de mostrar barras importantes? ¿Y su PACF?
- Con sus propias palabras, ¿por qué esos dos gráficos ayudan a elegir los parámetros p y q del modelo, en vez de adivinarlos?

Recuerden: la meta es que puedan explicarlo sin leer una definición, conectándolo con lo que ven en su propio gráfico.


## Celda 7 — Tabla AIC: comparando modelos automaticamente

In [ ]:
# Celda 7: En vez de adivinar, probamos varias combinaciones de (p, q)
# y nos quedamos con la que tenga el AIC mas bajo. Todo esto ya esta programado.

from itertools import product

mejor_aic = float("inf")
mejor_orden = None
resultados_aic = []

for p, q in product(range(3), range(3)):
    try:
        modelo_prueba = ARIMA(df["Ventas"], order=(p, d_sugerido, q)).fit()
        resultados_aic.append((p, d_sugerido, q, modelo_prueba.aic))
        if modelo_prueba.aic < mejor_aic:
            mejor_aic = modelo_prueba.aic
            mejor_orden = (p, d_sugerido, q)
    except Exception:
        pass

tabla_aic = pd.DataFrame(resultados_aic, columns=["p", "d", "q", "AIC"]).sort_values("AIC")
print(f"Tabla de comparacion de modelos para {SUCURSAL_ASIGNADA}:")
display(tabla_aic.head(6))
print()
print(f"Mejor combinacion encontrada: ARIMA{mejor_orden}")
print(f"AIC del mejor modelo: {mejor_aic:.2f}")
print()
print("Interpretacion: entre mas bajo el AIC, mejor el balance entre ajuste")
print("y simplicidad del modelo. Por eso elegimos automaticamente el mas bajo.")

### 🔎 INVESTIGUEN: ¿qué es el AIC y qué significa "el mejor modelo ARIMA"?

Investiguen: **¿qué es el AIC (Criterio de Información de Akaike) y por qué se usa para comparar modelos?** También: **¿qué significan las letras p, d y q en un modelo ARIMA?**

Ahora conecten esto con SU tabla de arriba:

- ¿Cuál fue la combinación (p, d, q) ganadora para su sucursal, y cuál fue su AIC exacto?
- Con sus propias palabras, ¿por qué esa combinación es "la mejor" y no otra de la tabla?

Esta explicación, con su modelo y su número de AIC, es una de las que más van a necesitar en el tríptico.


## Celda 8 — Entrenar el modelo final y pronosticar 6 meses

In [ ]:
# Celda 8: Ya con el mejor (p, d, q), ajustamos el modelo final
# y generamos el pronostico para los proximos 6 meses.

modelo_final = ARIMA(df["Ventas"], order=mejor_orden).fit()

pronostico = modelo_final.get_forecast(steps=6)
valores_pronostico = pronostico.predicted_mean
intervalo_confianza = pronostico.conf_int(alpha=0.05)

fechas_futuras = pd.date_range(df.index[-1] + pd.DateOffset(months=1), periods=6, freq="MS")

print(f"Pronostico de ventas a 6 meses - {SUCURSAL_ASIGNADA}")
print("=" * 55)
tabla_pronostico = pd.DataFrame({
    "Fecha": fechas_futuras.strftime("%Y-%m"),
    "Pronostico": valores_pronostico.values.round(0),
    "Limite inferior (95%)": intervalo_confianza.iloc[:, 0].values.round(0),
    "Limite superior (95%)": intervalo_confianza.iloc[:, 1].values.round(0),
})
display(tabla_pronostico)

In [ ]:
# Celda 8b: Grafico del historico + pronostico con intervalo de confianza

plt.figure(figsize=(12, 5.5))
plt.plot(df.index, df["Ventas"], label="Historico", color=COLOR_MORADO, linewidth=2)
plt.plot(fechas_futuras, valores_pronostico, label="Pronostico", color=COLOR_NARANJA,
         linewidth=2.5, linestyle="--", marker="o")
plt.fill_between(fechas_futuras, intervalo_confianza.iloc[:, 0], intervalo_confianza.iloc[:, 1],
                  color=COLOR_NARANJA, alpha=0.2, label="Intervalo de confianza 95%")
plt.title(f"Historico y pronostico - {SUCURSAL_ASIGNADA} ({provincia})", fontsize=14)
plt.xlabel("Fecha")
plt.ylabel("Ventas")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("Interpretacion: la linea naranja punteada es lo que el modelo cree que")
print("va a pasar. La franja sombreada es el rango donde es muy probable que")
print("caiga el valor real. Entre mas angosta la franja, mas confianza tiene")
print("el modelo en su propio pronostico.")

### 🔎 INVESTIGUEN: ¿qué es un intervalo de confianza en un pronóstico?

Investiguen: **¿qué es un intervalo de confianza del 95% y qué significa en el contexto de un pronóstico de series temporales?**

Ahora conecten esto con SU gráfico de arriba:

- ¿Qué tan ancha o angosta es la franja sombreada de su pronóstico? ¿Qué les dice eso sobre la certeza del modelo?
- Con sus propias palabras, ¿qué le explicarían a un gerente que nunca ha visto estadística, sobre qué significa ese rango sombreado?


## Celda 9 — ¿Que tan bueno es nuestro modelo?

In [ ]:
# Celda 9: Medimos el error del modelo comparando contra los ultimos datos reales
# (usamos los ultimos 6 meses del historico como si fueran "el futuro" ya conocido).

datos_entrenamiento = df["Ventas"][:-6]
datos_reales_prueba = df["Ventas"][-6:]

modelo_evaluacion = ARIMA(datos_entrenamiento, order=mejor_orden).fit()
pronostico_prueba = modelo_evaluacion.get_forecast(steps=6).predicted_mean

mape = mean_absolute_percentage_error(datos_reales_prueba, pronostico_prueba) * 100
rmse = np.sqrt(mean_squared_error(datos_reales_prueba, pronostico_prueba))

print(f"Resultados de precision del modelo - {SUCURSAL_ASIGNADA}")
print("=" * 55)
print(f"MAPE (error porcentual promedio): {mape:.1f}%")
print(f"RMSE (error en las mismas unidades que las ventas): {rmse:,.0f}")
print()

if mape < 10:
    calidad = "muy bueno"
elif mape < 20:
    calidad = "aceptable"
else:
    calidad = "con margen de mejora"

print(f"Interpretacion: un MAPE de {mape:.1f}% se considera un modelo {calidad}")
print("para decisiones de planificacion de inventario y personal.")
print()
print("PREGUNTA PARA EL GRUPO:")
print("Con este nivel de error, ¿confiarian en este modelo para decidir cuanto")
print("producto pedir el proximo mes en su sucursal? ¿Por que si o por que no?")

### 🔎 INVESTIGUEN: ¿qué son el MAPE y el RMSE?

Investiguen: **¿qué mide el MAPE, qué mide el RMSE, y en qué se diferencian uno del otro?**

Ahora conecten esto con SUS números de arriba:

- ¿Cuál fue el MAPE exacto de su modelo? Con sus propias palabras, ¿qué significa ese porcentaje para el negocio?
- ¿Cuál fue el RMSE? ¿En qué unidades está expresado y por qué es distinto al MAPE?

Estos dos números, explicados en sus propias palabras, son la base de la pregunta de confiabilidad que van a responder al final.


## Celda 10 — Resumen automatico de su caso

Esta celda arma un resumen con los numeros exactos de su sucursal. Van a necesitar estos numeros para el triptico.


In [ ]:
# Celda 10: Resumen ejecutivo automatico de esta sala.

print("=" * 60)
print(f"RESUMEN EJECUTIVO - {SUCURSAL_ASIGNADA} ({provincia})")
print("=" * 60)
print(f"Meses de historia analizados:        {len(df)}")
print(f"Venta promedio mensual:              {df['Ventas'].mean():,.0f}")
print(f"¿Serie estacionaria (antes de ajustar)?  {'Si' if es_estacionaria else 'No'}")
print(f"Modelo ARIMA elegido:                 {mejor_orden}")
print(f"AIC del modelo elegido:                {mejor_aic:.2f}")
print(f"MAPE de precision del modelo:          {mape:.1f}%")
print(f"RMSE del modelo:                       {rmse:,.0f}")
print(f"Pronostico primer mes futuro:           {valores_pronostico.values[0]:,.0f}")
print(f"Pronostico ultimo mes del semestre:     {valores_pronostico.values[-1]:,.0f}")
print("=" * 60)
print()
print("Guarden esta tabla de resultados, la van a usar directamente en su triptico.")

## Celda 11 — Preguntas finales para resolver EN GRUPO (no se programa nada mas)

Ya investigaron los 6 conceptos en el camino, pegados a cada resultado (las paradas que dicen INVESTIGUEN). Ahora, con todo eso fresco y con los resultados de arriba, respondan estas preguntas como grupo. Estas respuestas son la base de su triptico y de su presentación.

### Bloque A — Diagnóstico (entender lo que pasó)

**1.** ¿Qué sucursal les tocó, en qué provincia está, y cuántos meses de historia analizaron?

**2.** ¿Su serie era estacionaria o no? Expliquen el resultado de la prueba ADF con sus propias palabras, como si se lo explicaran a un compañero que faltó a clase.

**3.** ¿Qué le dijeron los gráficos ACF y PACF sobre su serie? ¿En qué lag dejaron de verse barras importantes?

**4.** ¿Qué combinación (p, d, q) ganó en la tabla AIC? ¿Por qué creen que esa combinación tuvo el menor AIC y no otra?

### Bloque B — Predictibilidad (qué tan bien podemos anticipar el futuro)

**5.** ¿Cuánto pronostica el modelo que va a vender su sucursal el próximo mes? ¿Y en el mes 6? ¿La tendencia es creciente, decreciente o estable?

**6.** ¿Qué tan ancho o angosto es el intervalo de confianza del 95% en su pronóstico? ¿Qué significa eso sobre la certeza del modelo?

**7.** Con el MAPE y el RMSE que les dio, ¿qué tan confiable es este modelo? ¿Confiarían en él para tomar una decisión real de negocio? Justifiquen con el número exacto, no con una opinión general.

### Bloque C — Predictibilidad aplicada a negocio (la decisión)

**8.** Si ustedes fueran el gerente de esta sucursal, ¿qué decisión de inventario tomarían para los próximos 6 meses con este pronóstico?

**9.** ¿Este modelo les ayudaría también a decidir sobre personal (más o menos empleados en ciertos meses) o sobre promociones? Den un ejemplo concreto con los números de su caso.

**10.** ¿Qué riesgo corre la sucursal si la gerencia ignora este pronóstico y solo se guía por intuición?

### Bloque D — Comparación entre casos

**11.** Cuando escuchen a otros grupos presentar: ¿en qué se parece o se diferencia su sucursal de la de ellos, en tendencia, estacionalidad, o nivel de error del modelo?

**12.** Si tuvieran que explicarle a alguien sin conocimientos de estadística, en una sola frase, qué hace un modelo ARIMA por el negocio, ¿qué le dirían?


## Guía para armar el tríptico

El tríptico NO es un trabajo aparte: es el ensamble de todo lo que ya fueron investigando y respondiendo en cada parada de "INVESTIGUEN" a lo largo del notebook, más las preguntas finales de la Celda 11. El tríptico debe tener 3 paneles:

**Panel 1 — El caso (diagnóstico):** nombre de la sucursal, provincia, el gráfico de la Celda 4, la pregunta de negocio que están resolviendo, y la respuesta a la pregunta 2 del Bloque A (estacionariedad, explicada con sus palabras, retomando lo que investigaron después de la Celda 5).

**Panel 2 — El análisis (conceptos + predictibilidad):** aquí va la parte más importante del tríptico, retomando directamente las paradas de investigación de las Celdas 6, 7, 8b y 9. Expliquen, con sus propias palabras y usando los números reales de su sucursal:
- Qué son ACF y PACF, y qué mostraron en su caso.
- Qué es la tabla AIC y cuál fue su mejor modelo ARIMA, y por qué.
- Qué es el pronóstico con intervalo de confianza del 95%, mostrando el gráfico de la Celda 8b.
- Qué son MAPE y RMSE, con los valores exactos de su modelo, y qué tan confiable es.

**Panel 3 — La decisión de negocio (predictibilidad aplicada):** la recomendación concreta de inventario, personal o promociones, y el riesgo de ignorar el modelo.

No hace falta un diseño complicado. Lo que se evalúa es que cualquiera que lea el tríptico entienda el caso, los conceptos explicados en palabras simples y conectados con sus propios números, el análisis y la decisión, no con frases genéricas copiadas de internet.
